## 2. Functions

### Default Arguments

In [ ]:
def f(a, L=[]):
    L.append(a)
    return L

print(f(1))
print(f(2))

### *args, **kwargs

In [ ]:
def my_sum(*args): # Packed into a single iterable object (tuple actually): args
    result = 0
    # Iterating over the Python args tuple
    for x in args:
        result += x
    return result

print(my_sum(1, 2, 3))

In [ ]:
def concatenate(**kwargs):
    """My Docstring"""
    result = ""
    # Iterating over the Python kwargs dictionary
    for arg in kwargs.values():
        result += arg
    return result

print(concatenate(a="Real", b="Python", c="Is", d="Great", e="!"))
print(concatenate.__doc__) # Retrieve function docstring

In [ ]:
'''def f(pos1, pos2, /, pos_or_kwd, *, kwd1, kwd2):
        -----------     ----------     ----------
        |                  |                  |
        |             Positional or keyword   |
        |                                     - Keyword only
        -- Positional only '''

'''The / is used to logically separate the positional-only parameters
from the rest of the parameters. Parameters following the / may be
positional-or-keyword or keyword-only'''
'''To mark parameters as keyword-only, place an * in the arguments list 
just before the first keyword-only parameter.'''

In [ ]:
'''When a final formal parameter of the form **name (**kwargs) is present, it receives a dictionary
containing all keyword arguments except for those corresponding to a formal parameter.'''
'''This may be combined with a formal parameter of the form *name (*args) which receives a tuple
containing the positional arguments beyond the formal parameter list. '''

### Functions as First-Class Objects

Functions can be passed as arguments, returned from other functions, assigned to variables and stored in data structures.

In [ ]:
def say_hello():
    print("Hello!")

say_hello

def salutation(salute):
    salute()

salutation(say_hello)
salutation(say_hello()) # NoneType Error

In [ ]:
print
print(say_hello)

funclist = [print, say_hello, salutation]
funclist

### Higher Order Functions

A function that takes a function as an argument or returns a function as the result is a *higher-order function*.

In [ ]:
def square(base):
    return base**2

list(map(square, [1, 2, 3, 4, 5]))

### Decorator Basics

A function returning another function, usually applied as a function transformation using the *@wrapper* syntax.

In [ ]:
import functools

def decorator(f):
    @functools.wraps(f) # functools preserves info about the original function: name, docstring, etc.
    # @functools.wraps() == functools.update_wrapper()
    def wrapper(*args, **kwds):
        print('Calling decorated function')
        return f(*args, **kwds)
    return wrapper

@decorator # == example = my_decorator(example)
def example():
    """Docstring"""
    print('Called example function')

example()
print(example.__name__, example.__doc__)


##### Nested Decorators

In [ ]:
import functools
import time

def logger(f):
    @functools.wraps(f)
    def wrapper_l(*args, **kwargs):
        args_repr = [repr(a) for a in args]
        kwargs_repr = [repr(k) for k in kwargs]
        print(f"Function name: {f.__name__} \nFunction arguments: {args_repr + kwargs_repr}")
        return f(*args, **kwargs)
    return wrapper_l

def timer(f):
    @functools.wraps(f)
    def wrapper_t(*args, **kwargs):
        start = time.perf_counter()
        value = f(*args, **kwargs)
        end = time.perf_counter()
        runtime = end - start
        print(f"Function executed in {runtime:.9f} seconds.")
        return value
    return wrapper_t

@logger
@timer # Equivalent to logger(timer(testfunc))
def testfunc(*args):
    print("Test Function working here!")
    return(len(args))
testfunc([x**2 for x in range(10)])

##### Defining Decorators with Arguments

In [ ]:
import functools

def repeat(num_times):
    def decorator_repeat(func):
        @functools.wraps(func)
        def wrapper_repeat(*args, **kwargs):
            for _ in range(num_times):
                value = func(*args, **kwargs)
            return value
        return wrapper_repeat
    return decorator_repeat

@repeat(num_times = 3)
def hello():
    print("Hello World!")

hello()

##### Decorators with Optional Arguments

In [ ]:
import functools

def repeat(_func=None, *, num_times=2):
    def decorator_repeat(func):
        @functools.wraps(func)
        def wrapper_repeat(*args, **kwargs):
            for _ in range(num_times):
                value = func(*args, **kwargs)
            return value
        return wrapper_repeat

    if _func is None:
        return decorator_repeat
    else:
        return decorator_repeat(_func)

@repeat()
def hello():
    print("Hello World!")
hello()

print('-'*15)

@repeat(num_times = 3)
def hello():
    print("Hello World!")
hello()

##### Decorators to Track States

In [ ]:
import functools

def count_calls(func):
    @functools.wraps(func)
    def wrapper_count_calls(*args, **kwargs):
        wrapper_count_calls.num_calls += 1
        print(f"Call {wrapper_count_calls.num_calls} of {func.__name__}()")
        return func(*args, **kwargs)
    wrapper_count_calls.num_calls = 0
    return wrapper_count_calls

@count_calls
def hi():
    print("Hi")

# Counting calls with the wrapper function attribute
hi() # First call  
hi() # Second call
hi() # ...
...


##### Caching Return Values

In [ ]:
import functools

def cache(func): # How to make a cache decorator
    """Keep a cache of previous function calls"""
    @functools.wraps(func)
    def wrapper_cache(*args, **kwargs):
        cache_key = args + tuple(kwargs.items())
        if cache_key not in wrapper_cache.cache:
            wrapper_cache.cache[cache_key] = func(*args, **kwargs)
        return wrapper_cache.cache[cache_key]
    wrapper_cache.cache = {}
    return wrapper_cache

@cache
@count_calls
def fibonacci(num):
    if num < 2:
        return num
    return fibonacci(num - 1) + fibonacci(num - 2)

# fibonacci(5)
# fibonacci(3)

# Use @functools.lru_cache or @functools.cache instead of my own cache decorator.



@functools.lru_cache(maxsize=4) # maxsize = None  is equivalent to using @functools.cache
def fibonacci(num):
    if num < 2:
        value = num
    else:
        value = fibonacci(num - 1) + fibonacci(num - 2)
    print(f"Calculated fibonacci({num}) = {value}")
    return value

# fibonacci(5)
# fibonacci(3)

### Closures

##### Scope

The scope of a variable in Python determines where in your code that variable is visible and accessible. 


Python has four general scope levels: local, enclosing, global, and built-in. When searching for a name, Python goes through these scopes in order. It follows the LEGB rule.


The scope of a name defines the region of a program where you can unambiguously access that name, which could identify a variable, constant, function, class, or any other object.

\- **Global scope:** Names in this scope are available to all your code.


\- **Local scope:** Names in this scope are only available or visible to the code within the scope.

##### LEGB Rule

Order in which the interpreter searches for a name through distinct scopes.

* **Local scope** is the body of any Python function or lambda expression. This scope contains the names that you define inside the function. These names are only visible from within the function.

* **Enclosing scope** is a scope that exists only for nested functions and is defined by the outer or enclosing function. This scope contains the names that you define within the enclosing function. The names in the enclosing scope are visible from the code of the inner and outer functions.

* **Global scope** is the topmost scope. This scope contains all of the names that you define at the top level of a script or module. Names in this scope are visible from everywhere in your code.

* **Built-in scope** is a special scope that Python creates or loads whenever you run a script. This scope contains names such as built-in functions, exceptions, and other attributes that are built into Python. Names in this scope are also available from everywhere in your code.


L -> E -> G -> B sequentially.  If Python finds the name, then you’ll get its first or innermost occurrence. Otherwise, you’ll get a NameError exception.

##### Local Scope

Python creates a local scope for every function call. The arguments that you pass to a function and the variables that you define inside a function exist only within the function’s local scope. When the function returns, this local scope is destroyed.

In [ ]:
def square(base):
   result = base**2
   print(dir()) # dir() returns variable names
   print(vars()) # vars() returns a dict of {vars: values}
   print(f"The square of {base} is: {result}")

square(1)
result # Only accesible inside square()

##### Enclosing (nonlocal) Scope

Created when a function is nested inside another function. Names that you define in the enclosing scope are known as ``nonlocal`` names because they’re neither local nor global. 


The enclosing scope is accessible to the nested functions during their execution.

In [ ]:
def outer_func():
    # The code block of outer_func() defines its local scope
    # It also defines the enclosing scope of inner_func()
    variable = 100  # Local to outer_func() and nonlocal to inner_func()
    def inner_func():
        # The code block of inner_func() defines its local scope
        print(f"Printing variable from inner_func(): {variable}")

    inner_func()
    print(f"Printing variable from outer_func(): {variable}")


outer_func()
inner_func() # not defined since it can only be accessed from from outer_func()


All the names you create in the enclosing scope are visible from inside inner_func(), except for those created after you call inner_func().

##### Global Scope

Internally, Python turns your program’s main script into a module called `__main__` to hold the main program’s execution. The namespace of this module represents the global scope of your program. This module or script is loaded with the special name, ``__main__``. From this point on, the scope of ``__main__`` is the program’s global scope.

In [ ]:
var = 1
dir() # Inspect the names of the global scope

When the program terminates, the global scope is discarded, and its names are forgotten. You can access or reference the value of any global name from any place in your code.

In [ ]:
number = 3

def get_number():
    return number

get_number()

Inside a function, if you assign a value to a variable that exists in the global scope, then Python creates a new local variable with the same name instead of modifying the global variable.

**Note:** You can update or modify global names from any place within your current global scope. To update them from the local scope, you can use the ``global`` statement.

##### Built-in Scope

Special scope that’s implemented as a standard library module called ``builtins``. Python searches builtins last in its LEGB lookup sequence.

The names in builtins are loaded into your global scope as a mapping with the special name ``__builtins__``.

In [ ]:
list(__builtins__.__dict__.keys()) # Whole list of Python's builtin objects

In [ ]:
import builtins # Import the standard Python's module. Loaded into global.

dir(builtins)

##### Modifying Scope behaviour

* ``global`` Statement

In [ ]:
counter = 0

def update_counter():
    global counter # Declares 'counter' as a global variable. Tells Python to look for it in the global scope
    counter =+ 1 # Updates it
    
update_counter()

## In general, considered a **BAD** practice

In [ ]:
# Creating global names LAZILY: (DANGEROUS!)

def create_lazy():
    global numb # Declare the variable 'number' as global
    numb = 5    # Create a global variable lazily
    return numb

create_lazy()
numb # Now available in the global scope

* ``nonlocal`` Statement

In [ ]:
# Modify a variable declared in the nonlocal/enclosing scope

def function():
    number = 42  # A nonlocal variable
    def nested():
        nonlocal number  # Declare 'number' as nonlocal
        number += 42
    nested()
    print(number)

function()

# Cannot create nonlocal names lazily.

##### Creating **Closures** with the Enclosing Scope

Closure: function object that has access to variables in its enclosing scope. A closure occurs when a nested function references variables from its enclosing function. The nested function “closes over” the data, maintaining a reference to it.

In [ ]:
def outer_function(x):
    """Enclose the value of x in the closure scope."""
    def inner_function(y):
        # x is a free variable captured from the enclosing scope
        return x + y
    return inner_function

# Create a closure and store it 
add_five = outer_function(5)

# Call the closure function
result = add_five(3)
print(result)  # Output

##### Bringing names to Scope: ``import``

Load the names defined in any module into your global scope.

In [ ]:
dir()

In [ ]:
import os
dir()

##### Other Scope Behaviours

* Comprehensions:

    The loop variable in a comprehension is local to the comprehension itself

In [ ]:
[number**2 for number in range(5)]
number

* Exceptions

* Classes, Instances

##### Scope-Related Built-in Functions

* `globals()`

In [ ]:
globals() # Returns a dict with all the names and its assigned objects currently in the global scope.

* `locals()`

In [ ]:
def function(arg):
    var = 100
    print(locals()) # Returns a dict with a copy of the current state of the local scope.
    # Returns dict with all the names and its assigned objects in the local scope down to where
    # locals() is called.
    another = 200

function(arg = 5)

In [ ]:
locals() # Equivalent to globals() if called in the global scope.
locals() is globals()